# CropDetector — Phase 9: Input Validation (Part 1)
### Reproducible dataset audit and persistent manifests

**Independent notebook:** Does **not** need variables from the 130 earlier cells, the old Keras checkpoint, or re-training the 15-class classifier.

**This first part:** downloads **Cropped PlantDoc**; labels *supported crop/condition*, *unsupported crop*, and *unsupported condition*; checks exact and perceptual duplicates; makes leakage-aware development partitions; exports a ZIP with manifest CSVs and an audit report to preserve on your Mac.

**Not yet trained:** A reliable *non-plant* detector. Non-plant images are **not** available in PlantDoc, so **do not** label unsupported plants as non-plant objects. Phase 9 Part 2 will add diverse non-plant photographs, train the validator, and export INT8.

**Evaluation caveat:** We used PlantDoc test images repeatedly during Phase 5–8. Any tests on PlantDoc test are *diagnostics*, **not an untouched independent final test**. Reserve new field images for final verification.

## Cell 1 — Install dependencies and initialize paths
Use **Runtime → Run all** in a fresh Google Colab notebook. A GPU is **not required** for this audit.

In [ ]:
%pip -q install imagehash

from pathlib import Path
from PIL import Image, ImageOps, UnidentifiedImageError
from IPython.display import display
import pandas as pd
import numpy as np
import hashlib, imagehash, subprocess, json, zipfile, shutil

SEED = 42
PHASH_THRESHOLD = 4  # conservative candidate grouping, not proof of identical source
ROOT = Path('/content/PlantDoc-Dataset')
OUT = Path('/content/CropDetector_Phase9_Audit')
OUT.mkdir(parents=True, exist_ok=True)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
print('Ready. Audit output directory:', OUT)

## Cell 2 — Download the exact PlantDoc classification repository
The source contains separate `train/` and `test/` folders. We record the Git commit for reproducibility.

In [ ]:
URL = 'https://github.com/pratikkayal/PlantDoc-Dataset.git'
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(ROOT)], check=True)

COMMIT = subprocess.check_output(
    ['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True
).strip()

for part in ('train', 'test'):
    assert (ROOT / part).is_dir(), f'Missing {part} folder'
    n = sum(p.suffix.lower() in IMG_EXT for p in (ROOT / part).rglob('*') if p.is_file())
    print(f'PlantDoc {part:>5}: {n:,} images')

print('Repository commit:', COMMIT)

## Cell 3 — Recreate the exact known-class mapping
The 15-class classifier is trained for **Maize, Grape, Potato and Tomato**. PlantDoc covers only 10 of the 15 classification labels. We preserve that mapping and exclude ambiguous `Corn leaf blight` rather than labelling it as unknown.

In [ ]:
CLASS_NAMES = [
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___healthy',
]
SUPPORTED_MAPPING = {
    'corn gray leaf spot': 0,
    'corn rust leaf': 1,
    'grape leaf black rot': 4,
    'grape leaf': 7,
    'potato leaf early blight': 8,
    'potato leaf late blight': 9,
    'potato leaf': 10,
    'tomato leaf bacterial spot': 11,
    'tomato early blight leaf': 12,
    'tomato leaf late blight': 13,
    'tomato leaf': 14,
}
UNSUPPORTED_CROP_PREFIXES = (
    'apple', 'bell_pepper', 'blueberry', 'cherry', 'peach',
    'raspberry', 'soyabean', 'soybean', 'squash', 'strawberry'
)

def label_folder(folder):
    name = folder.strip().lower()
    if name in SUPPORTED_MAPPING:
        return 'Supported', SUPPORTED_MAPPING[name]
    if name == 'corn leaf blight':
        return 'Excluded / ambiguous', -1
    if name.startswith('tomato'):
        return 'Unsupported Condition', -1
    if name.startswith(UNSUPPORTED_CROP_PREFIXES):
        return 'Unsupported Crop', -1
    return 'Excluded / review', -1

assert len(CLASS_NAMES) == 15
print('Supported classification labels:', len(CLASS_NAMES))

## Cell 4 — Build a deterministic image manifest
Keep **relative paths** so the manifest is portable across fresh Colab runtimes. We also record the full runtime paths in memory.

In [ ]:
rows = []
for split in ('train', 'test'):
    for folder in sorted((ROOT / split).iterdir()):
        if not folder.is_dir():
            continue
        role, label = label_folder(folder.name)
        for path in sorted(folder.rglob('*')):
            if path.is_file() and path.suffix.lower() in IMG_EXT:
                rows.append({
                    'Split': split,
                    'Folder': folder.name,
                    'Role': role,
                    'Label': label,
                    'RelativePath': path.relative_to(ROOT).as_posix(),
                    'Path': str(path),
                })

manifest = pd.DataFrame(rows).sort_values('RelativePath').reset_index(drop=True)
assert not manifest.empty
print('Total image files:', len(manifest))
display(manifest.groupby(['Split','Role']).size().rename('Images').to_frame())
print('Folders requiring manual review:')
display(manifest[manifest.Role.str.startswith('Excluded')].groupby('Folder').size())

## Cell 5 — Compute SHA-256 and perceptual hashes
SHA-256 detects byte-identical files; 64-bit pHash catches similar-looking images and transformed duplicates. pHash matches are **candidates**, not proof of the same original photograph. We conservatively group them for leakage screening.

In [ ]:
def fingerprint(path):
    payload = Path(path).read_bytes()
    exact = hashlib.sha256(payload).hexdigest()
    with Image.open(path) as im:
        rgb = ImageOps.exif_transpose(im).convert('RGB')
        ph = int(str(imagehash.phash(rgb, hash_size=8)), 16)
    return exact, ph

shas, hashes, errors = [], [], []
for i, path in enumerate(manifest.Path):
    try:
        sha, ph = fingerprint(path)
        shas.append(sha)
        hashes.append(ph)
        errors.append('')
    except (OSError, ValueError, UnidentifiedImageError) as exc:
        shas.append('')
        hashes.append(None)
        errors.append(f'{type(exc).__name__}: {exc}')

manifest['SHA256'] = shas
manifest['pHash'] = [f'{v:016x}' if v is not None else '' for v in hashes]
manifest['ReadError'] = errors
print('Unreadable files:', (manifest.ReadError != '').sum())
if (manifest.ReadError != '').any():
    display(manifest.loc[manifest.ReadError != '', ['RelativePath','ReadError']])
manifest = manifest[manifest.ReadError == ''].copy().reset_index(drop=True)

## Cell 6 — Group exact and near-duplicate images
This checks duplicates **across both original train and test**; a group that spans the two partitions must not leak into training. Pairwise 64-bit Hamming screening is adequate for the ~2.6k-image PlantDoc repository.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, a):
        while self.parent[a] != a:
            self.parent[a] = self.parent[self.parent[a]]
            a = self.parent[a]
        return a
    def union(self, a, b):
        x, y = self.find(a), self.find(b)
        if x == y: return
        if self.rank[x] < self.rank[y]: x, y = y, x
        self.parent[y] = x
        if self.rank[x] == self.rank[y]: self.rank[x] += 1

u = UnionFind(len(manifest))
sha_first = {}
for i, sha in enumerate(manifest['SHA256']):
    if sha in sha_first: u.union(i, sha_first[sha])
    else: sha_first[sha] = i

pvals = [int(v,16) for v in manifest['pHash']]
near_pairs = 0
for i in range(len(pvals)):
    for j in range(i + 1, len(pvals)):
        if (pvals[i] ^ pvals[j]).bit_count() <= PHASH_THRESHOLD:
            near_pairs += 1
            u.union(i, j)

manifest['Group'] = [f'pd_{u.find(i):05d}' for i in range(len(manifest))]
groups = manifest.groupby('Group')
conflict_groups = set(groups.filter(lambda x: x.Folder.nunique() > 1).Group)
overlap_groups = set(groups.filter(lambda x: x.Split.nunique() > 1).Group)
multi_groups = int((groups.size() > 1).sum())
print('Exact duplicate extra files:', len(manifest) - manifest.SHA256.nunique())
print('pHash candidate pairs (distance <= 4):', near_pairs)
print('Multi-image duplicate groups:', multi_groups)
print('Conflicting folder groups:', len(conflict_groups))
print('Train/test overlapping groups:', len(overlap_groups))

## Cell 7 — Leakage-aware train/dev partition
Exclude conflicting-label groups and every group spanning original train/test from **both sides**. Create a deterministic folder-stratified 80/20 **development** split using the original `train` portion only.

> This is a *new split*, not a reconstruction of the previous Phase 5 617/155 rows. That old split would require the exact saved CSV. We will not claim the splits are identical.

In [ ]:
unsafe_groups = conflict_groups | overlap_groups
clean = manifest[
    ~manifest.Group.isin(unsafe_groups)
    & ~manifest.Role.str.startswith('Excluded')
].copy()

rng = np.random.default_rng(SEED)
dev_groups = set()
for (role,folder), part in clean[clean.Split == 'train'].groupby(['Role','Folder']):
    group_ids = sorted(part.Group.unique())
    if len(group_ids) < 2:
        continue
    count = max(1, round(0.2 * len(group_ids)))
    count = min(count, len(group_ids)-1)
    dev_groups.update(rng.choice(group_ids, size=count, replace=False).tolist())

train_df = clean[(clean.Split == 'train') & (~clean.Group.isin(dev_groups))].copy()
dev_df = clean[(clean.Split == 'train') & (clean.Group.isin(dev_groups))].copy()
legacy_test_df = clean[clean.Split == 'test'].copy()

assert set(train_df.Group).isdisjoint(dev_df.Group)
assert set(train_df.Group).isdisjoint(legacy_test_df.Group)
assert set(dev_df.Group).isdisjoint(legacy_test_df.Group)
assert not (clean.Role == 'Excluded / review').any()

print('Train image count:', len(train_df))
print('Development image count:', len(dev_df))
print('Previously-used PlantDoc test count:', len(legacy_test_df))
print('Removed ambiguous/unsafe image files:', len(manifest)-len(clean))

inventory = pd.concat([
    train_df.assign(Partition='train'),
    dev_df.assign(Partition='development'),
    legacy_test_df.assign(Partition='legacy_test_diagnostic'),
])
display(inventory.groupby(['Partition','Role']).size().rename('Images').to_frame())

## Cell 8 — Inspect important limitations
A separate *non-plant* dataset is required before a trustworthy validator can be trained. The cropped PlantDoc images are mainly leaves. Unsupported-disease images are also **not** equivalent to non-plant images.

In [ ]:
print('Supported known-class coverage in PlantDoc train:')
display(train_df[train_df.Role=='Supported'].groupby(['Label','Folder']).size().rename('Images').to_frame())
print('\nUnknown crop/condition development coverage:')
display(dev_df[dev_df.Role!='Supported'].groupby(['Role','Folder']).size().rename('Images').to_frame())

quality_checks = {
 'no_train_dev_group_overlap': set(train_df.Group).isdisjoint(dev_df.Group),
 'no_train_legacy_test_group_overlap': set(train_df.Group).isdisjoint(legacy_test_df.Group),
 'nonplant_examples_available': False,
 'new_independent_field_test_available': False,
}
print('\nData quality checks:')
for key,val in quality_checks.items(): print(f'  {key}: {val}')

## Cell 9 — Save permanent manifests and download a ZIP
**Always download this ZIP to your computer.** Colab `/content` is temporary. This ZIP contains the audit data and paths; it does **not** contain the image dataset. The notebook can clone the images again in a future session.

In [ ]:
for name, df in [
    ('plantdoc_full_audit.csv',manifest),
    ('plantdoc_train.csv',train_df),
    ('plantdoc_dev.csv',dev_df),
    ('plantdoc_legacy_test_diagnostic.csv',legacy_test_df),
]:
    export = df.drop(columns=['Path'], errors='ignore')
    export.to_csv(OUT/name, index=False)

report = {
    'project':'CropDetector Phase 9 Part 1',
    'plantdoc_source':URL,
    'git_commit':COMMIT,
    'seed':SEED,
    'phash_hamming_threshold':PHASH_THRESHOLD,
    'source_images':len(manifest),
    'train_images':len(train_df),
    'development_images':len(dev_df),
    'legacy_test_diagnostic_images':len(legacy_test_df),
    'cross_split_duplicate_groups':len(overlap_groups),
    'conflicting_label_duplicate_groups':len(conflict_groups),
    'known_development_images':int((dev_df.Role=='Supported').sum()),
    'unsupported_crop_development_images':int((dev_df.Role=='Unsupported Crop').sum()),
    'unsupported_condition_development_images':int((dev_df.Role=='Unsupported Condition').sum()),
    'nonplant_dataset_added':False,
    'important_note':'PlantDoc test was previously inspected and is not an untouched final test.',
}
(OUT/'audit_report.json').write_text(json.dumps(report,indent=2))
(OUT/'README.txt').write_text(
    'CropDetector Phase 9 audit: generated from the public PlantDoc classification repo.\n'
    'CSV paths are relative to the cloned PlantDoc repository root.\n'
    'Re-clone the same source commit in future Colab sessions.\n'
    'Legacy test is for diagnostics only; it has already influenced project development.\n'
    'Non-plant data and independent field test still need to be added.\n'
)
archive = shutil.make_archive('/content/CropDetector_Phase9_Audit', 'zip', OUT)
print('Saved report:', json.dumps(report, indent=2))
print('Archive:', archive)
from google.colab import files
files.download(archive)

## Next: Phase 9 Part 2
After verifying the audit, we will add real, diverse **non-plant** photographs (with provenance and split controls), train a small **plant/leaf input validity** model, and evaluate *supported crop vs unsupported crop* separately. Then we will quantize the validator and test it on Exynos 1330.

**Please share:** the output of Cell 7, Cell 8, and the `audit_report.json` file (or the downloaded audit ZIP). There is no need to run any of the old 130 cells.